# SMITH human disease transfer robustness

Participant-level human brain data are controlled access. This notebook therefore performs an auditable analysis of the released de-identified aggregate table: it compares improvements in rank correlation and top-64 panel overlap across cohort transfers.

[Open the editable source notebook on GitHub](https://github.com/fym0503/SMITH/blob/main/docs/source/tutorials/notebooks/disease_section/04_SMITH_InHouse_Disease_Transfer_source.ipynb)

## Setup

Run this notebook from a cloned SMITH repository with `pip install -e '.[notebooks]'`. All inputs are checksum-validated before analysis.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_repository(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "reproducibility").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside a SMITH repository checkout.")


ROOT = find_repository(Path.cwd().resolve())
sys.path.insert(0, str(ROOT / "src"))

from smith.reproducibility import check_case, load_cases, run_case

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 30)
print(f"Repository: {ROOT}")


In [ ]:
CASE_ID = "04_inhouse_disease"
case = load_cases()[CASE_ID]
status = check_case(case)
if status["inputs"]:
    display(pd.DataFrame(status["inputs"])[["path", "exists", "sha256_ok"]])
else:
    print("This tutorial creates its deterministic input during execution.")
assert status["ready"], "The pinned tutorial inputs are missing or have changed."

output_dir = ROOT / "outputs" / "notebooks" / CASE_ID
result = run_case(case, output_dir)
print(f"Summary written to: {result['summary_json']}")
result


## Analysis

In [ ]:
robustness = pd.DataFrame(result["transfer_robustness"])
display(robustness)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(robustness["comparison"], robustness["delta_spearman_mean"], color="#218c74")
axes[0].set(ylabel="Δ Spearman correlation", title="Expression-rank transfer")
axes[1].bar(robustness["comparison"], robustness["delta_top64_mean"], color="#cc8e35")
axes[1].set(ylabel="Δ top-64 overlap", title="Panel-overlap transfer")
for ax in axes:
    ax.tick_params(axis="x", rotation=45)
    ax.axhline(0, color="black", linewidth=0.8)
fig.tight_layout()
plt.show()


## Data boundary

The notebook reproduces only analyses supported by the de-identified aggregate release. Participant-level panels, UMAPs, imputation results, and spatial gene examples require authorized inputs; this is recorded explicitly in `reproducibility/manifests/04_inhouse_disease.yaml`.